# Retail Sales Forecasting - Data Science Analysis\n\nThis notebook performs exploratory data analysis (EDA) and time series forecasting on the Superstore Sales dataset using LSTM and GRU neural networks.\n\n**Sections:**\n1. Data Loading & EDA\n2. Feature Extraction (cyclical encoding)\n3. Data Scaling (StandardScaler)\n4. Train/Validation/Test Split\n5. LSTM Model\n6. GRU Model\n7. Results Comparison & Visualization

## 1. Data Loading & Exploratory Data Analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.callbacks import EarlyStopping

In [ ]:
# Load all 4 chunks (simulating loading from the data warehouse)
chunks = []
for i in range(1, 5):
    chunk = pd.read_csv(f'../data/split/sales_chunk_{i}.csv')
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Parse dates and create daily aggregated sales
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
daily_sales = df.groupby('Order Date').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Quantity=('Quantity', 'sum'),
    OrderCount=('Order ID', 'nunique')
).reset_index()
daily_sales.columns = ['Date', 'Sales', 'Profit', 'Quantity', 'OrderCount']
daily_sales.set_index('Date', inplace=True)
daily_sales = daily_sales.asfreq('D', fill_value=0)
print(f"Daily sales rows: {len(daily_sales)}")
print(f"Date range: {daily_sales.index.min()} to {daily_sales.index.max()}")
daily_sales.head(10)

### EDA: Sales Distribution & Trends

In [ ]:
# Daily sales over time
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(daily_sales.index, daily_sales['Sales'], linewidth=0.5)
axes[0, 0].set_title('Daily Sales Over Time')
axes[0, 0].set_ylabel('Sales ($)')

axes[0, 1].plot(daily_sales.index, daily_sales['Profit'], linewidth=0.5, color='green')
axes[0, 1].set_title('Daily Profit Over Time')
axes[0, 1].set_ylabel('Profit ($)')

# Monthly aggregation for smoother view
monthly = daily_sales.resample('M').sum()
axes[1, 0].bar(monthly.index, monthly['Sales'], width=20, color='steelblue')
axes[1, 0].set_title('Monthly Sales')
axes[1, 0].set_ylabel('Sales ($)')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].hist(daily_sales['Sales'][daily_sales['Sales'] > 0], bins=50, color='orange', edgecolor='black')
axes[1, 1].set_title('Sales Distribution (non-zero days)')
axes[1, 1].set_xlabel('Daily Sales ($)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Day-of-week pattern
dow_sales = daily_sales.copy()
dow_sales['DayOfWeek'] = dow_sales.index.dayofweek
dow_avg = dow_sales.groupby('DayOfWeek')['Sales'].mean()
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

plt.figure(figsize=(8, 4))
plt.bar(day_names, dow_avg.values, color='steelblue')
plt.title('Average Daily Sales by Day of Week')
plt.ylabel('Average Sales ($)')
plt.show()

In [ ]:
# Basic statistics
print("=== Daily Sales Statistics ===")
print(daily_sales.describe())
print(f"\nTotal revenue: ${daily_sales['Sales'].sum():,.2f}")
print(f"Total profit: ${daily_sales['Profit'].sum():,.2f}")
print(f"Profit margin: {daily_sales['Profit'].sum() / daily_sales['Sales'].sum() * 100:.1f}%")

## 2. Feature Extraction\n\nWe encode cyclical time features using sine/cosine transformations, which allows the model to understand the periodic nature of day-of-week and day-of-year patterns.

In [ ]:
# Cyclical encoding of time features
daily_sales['DayOfWeek'] = daily_sales.index.dayofweek
daily_sales['DayOfYear'] = daily_sales.index.dayofyear

# Sine/cosine encoding (same approach as HF4)
daily_sales['DoW_X'] = np.sin(2 * np.pi * daily_sales['DayOfWeek'] / 7)
daily_sales['DoW_Y'] = np.cos(2 * np.pi * daily_sales['DayOfWeek'] / 7)
daily_sales['DoY_X'] = np.sin(2 * np.pi * daily_sales['DayOfYear'] / 365)
daily_sales['DoY_Y'] = np.cos(2 * np.pi * daily_sales['DayOfYear'] / 365)

# Drop raw cyclical columns
daily_sales.drop(['DayOfWeek', 'DayOfYear'], axis=1, inplace=True)

print("Features after extraction:")
print(daily_sales.columns.tolist())
daily_sales.head()

## 3. Data Scaling & 4. Train/Validation/Test Split\n\nWe split the data by time periods and apply StandardScaler (fitted on training data only) to normalize all features.

In [ ]:
# Time-based split
train_start = pd.Timestamp("2014")
valid_start = pd.Timestamp("2017")
test_start = pd.Timestamp("2018")

train_df = daily_sales[(daily_sales.index >= train_start) & (daily_sales.index < valid_start)].copy()
valid_df = daily_sales[(daily_sales.index >= valid_start) & (daily_sales.index < test_start)].copy()
test_df = daily_sales[(daily_sales.index >= test_start)].copy()

print(f"Train: {len(train_df)} days ({train_df.index.min()} to {train_df.index.max()})")
print(f"Valid: {len(valid_df)} days ({valid_df.index.min()} to {valid_df.index.max()})")
print(f"Test:  {len(test_df)} days ({test_df.index.min()} to {test_df.index.max()})")

In [ ]:
# Scaling with StandardScaler
scaler_input = StandardScaler()
scaler_output = StandardScaler()

target_col = "Sales"

scaled_train = scaler_input.fit_transform(train_df)
target_train = scaler_output.fit_transform(train_df[[target_col]])
scaled_valid = scaler_input.transform(valid_df)
target_valid = scaler_output.transform(valid_df[[target_col]])
scaled_test = scaler_input.transform(test_df)
target_test = scaler_output.transform(test_df[[target_col]])

print(f"Scaled train shape: {scaled_train.shape}")
print(f"Input mean (train): {scaled_train.mean(axis=0).round(3)}")
print(f"Input std  (train): {scaled_train.std(axis=0).round(3)}")

### Sequence Creation\n\nWe create input sequences of `lookback` days for the LSTM/GRU models to learn temporal patterns.

In [ ]:
def process_Xy(raw_X: np.array, raw_y: np.array, lookback: int) -> tuple:
    """Create sequences of length `lookback` for time series modeling."""
    X = np.empty(shape=(raw_X.shape[0] - lookback, lookback, raw_X.shape[1]), dtype=np.float32)
    y = np.empty(shape=(raw_y.shape[0] - lookback), dtype=np.float32)

    target_index = 0
    for i in range(lookback, raw_X.shape[0]):
        X[target_index] = raw_X[i - lookback : i]
        y[target_index] = raw_y[i].item()
        target_index += 1

    return X.copy(), y.copy()

lookback = 14

train_X, train_y = process_Xy(scaled_train, target_train, lookback=lookback)
valid_X, valid_y = process_Xy(scaled_valid, target_valid, lookback=lookback)
test_X, test_y = process_Xy(scaled_test, target_test, lookback=lookback)

print(f"Train X: {train_X.shape}, Train y: {train_y.shape}")
print(f"Valid X: {valid_X.shape}, Valid y: {valid_y.shape}")
print(f"Test  X: {test_X.shape}, Test  y: {test_y.shape}")

## 5. LSTM Model

In [ ]:
# Build LSTM model
model_lstm = keras.Sequential([
    layers.LSTM(16, activation="relu", input_shape=train_X.shape[1:]),
    layers.Dense(1),
])

model_lstm.compile(loss='MeanSquaredError', optimizer='Adam')
model_lstm.summary()

In [ ]:
# Train LSTM
callbacks_lstm = [EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]

history_lstm = model_lstm.fit(
    train_X, train_y,
    validation_data=(valid_X, valid_y),
    batch_size=16,
    epochs=100,
    callbacks=callbacks_lstm,
    shuffle=True,
    verbose=True,
)

In [ ]:
# LSTM Training/Validation Loss
plt.figure(figsize=(8, 4))
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.title('LSTM - Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

In [ ]:
# LSTM Predictions on Test Set
pred_lstm = model_lstm.predict(test_X)

plt.figure(figsize=(12, 5))
plt.plot(test_df["Sales"], label="Actual", color='blue', linewidth=0.8)
plt.plot(pd.DataFrame(
    index=test_df.index[lookback:],
    data=scaler_output.inverse_transform(pred_lstm)
), label="LSTM Predicted", color='red', linewidth=0.8)
plt.title('LSTM - Actual vs Predicted Daily Sales (Test Set)')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. GRU Model\n\nWe train the same architecture with a GRU layer instead of LSTM for comparison.

In [ ]:
# Build GRU model
model_gru = keras.Sequential([
    layers.GRU(16, activation="relu", input_shape=train_X.shape[1:]),
    layers.Dense(1),
])

model_gru.compile(loss='MeanSquaredError', optimizer='Adam')
model_gru.summary()

In [ ]:
# Train GRU
callbacks_gru = [EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]

history_gru = model_gru.fit(
    train_X, train_y,
    validation_data=(valid_X, valid_y),
    batch_size=16,
    epochs=100,
    callbacks=callbacks_gru,
    shuffle=True,
    verbose=True,
)

In [ ]:
# GRU Training/Validation Loss
plt.figure(figsize=(8, 4))
plt.plot(history_gru.history['loss'], label='Training Loss')
plt.plot(history_gru.history['val_loss'], label='Validation Loss')
plt.title('GRU - Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

In [ ]:
# GRU Predictions on Test Set
pred_gru = model_gru.predict(test_X)

plt.figure(figsize=(12, 5))
plt.plot(test_df["Sales"], label="Actual", color='blue', linewidth=0.8)
plt.plot(pd.DataFrame(
    index=test_df.index[lookback:],
    data=scaler_output.inverse_transform(pred_gru)
), label="GRU Predicted", color='red', linewidth=0.8)
plt.title('GRU - Actual vs Predicted Daily Sales (Test Set)')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Results Comparison & Visualization\n\nCompare LSTM vs GRU performance side-by-side on the test set.

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

test_dates = test_df.index[lookback:]
actual_test = test_df["Sales"].iloc[lookback:]
pred_lstm_inv = scaler_output.inverse_transform(pred_lstm).flatten()
pred_gru_inv = scaler_output.inverse_transform(pred_gru).flatten()

axes[0].plot(test_dates, actual_test, label="Actual", color='blue', linewidth=0.7)
axes[0].plot(test_dates, pred_lstm_inv, label="LSTM", color='red', linewidth=0.7)
axes[0].set_title('LSTM Predictions')
axes[0].set_ylabel('Sales ($)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(test_dates, actual_test, label="Actual", color='blue', linewidth=0.7)
axes[1].plot(test_dates, pred_gru_inv, label="GRU", color='orange', linewidth=0.7)
axes[1].set_title('GRU Predictions')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('LSTM vs GRU — Daily Sales Forecast Comparison (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse_lstm = mean_squared_error(actual_test, pred_lstm_inv)
mae_lstm = mean_absolute_error(actual_test, pred_lstm_inv)
mse_gru = mean_squared_error(actual_test, pred_gru_inv)
mae_gru = mean_absolute_error(actual_test, pred_gru_inv)

print("=" * 50)
print("Model Performance on Test Set")
print("=" * 50)
print(f"{'Metric':<15} {'LSTM':>12} {'GRU':>12}")
print("-" * 40)
print(f"{'MSE':<15} {mse_lstm:>12,.2f} {mse_gru:>12,.2f}")
print(f"{'RMSE':<15} {np.sqrt(mse_lstm):>12,.2f} {np.sqrt(mse_gru):>12,.2f}")
print(f"{'MAE':<15} {mae_lstm:>12,.2f} {mae_gru:>12,.2f}")
print("=" * 50)

best = "LSTM" if mse_lstm < mse_gru else "GRU"
print(f"\nBetter model (lower MSE): {best}")

## Summary\n\nThis analysis covered all required data science components:\n\n1. **EDA** — Distribution, trends, day-of-week patterns using pandas + matplotlib\n2. **Feature Extraction** — Cyclical sine/cosine encoding of day-of-week and day-of-year\n3. **Scaling** — StandardScaler applied to inputs and target variable\n4. **Train/Validation/Test Split** — Time-based: 2014-2016 / 2017 / 2018\n5. **LSTM Model** — Keras Sequential LSTM(16) + Dense(1) with EarlyStopping\n6. **GRU Model** — Same architecture with GRU layer for comparison\n7. **Visualization** — Loss curves, actual vs predicted plots, quantitative metrics (MSE, RMSE, MAE)